# UI-PRMD 위치 기반 TCN — Colab 학습

목표: M01 Deep Squat, M03 Inline Lunge, M04 Side Lunge, M06 Standing Active Straight Leg Raise, M07 Standing Shoulder Abduction의 **동작 종류** 분류. 자세의 정확성/재활 품질 점수가 아닙니다.

Colab에서 **런타임 → 런타임 유형 변경 → GPU**를 선택하고 위에서부터 실행하세요. Movements.zip을 Google Drive에 올리고 아래 경로를 수정합니다. 모델은 실행 후 생성되며 이 노트북에는 사전 학습 결과가 없습니다.

### 데이터 선택과 중요한 좌표 차이
이 폴더의 Vicon Positions는 39개 광학 마커의 XYZ 궤적(117열)입니다. Kinect Positions는 루트를 제외하면 부모 좌표계의 상대 위치라서 단순 합산만으로 MediaPipe와 대응시킬 수 없습니다. **Angles를 읽지 않는 위치 전용 baseline**을 위해 Vicon을 사용합니다. Vicon의 마커 위치는 해부학적 관절 중심과 정확히 같지 않습니다. 특히 골반 중심은 아래 마커 평균으로 근사합니다.

|공통 관절 순서|Vicon 마커 (1-based)|MediaPipe (0-based)|
|---|---|---|
|왼/오른 어깨|10 / 17|11 / 12|
|왼/오른 팔꿈치|12 / 19|13 / 14|
|왼/오른 손목|평균(14,15) / 평균(21,22)|15 / 16|
|왼/오른 골반|평균(24,26) / 평균(25,27)|23 / 24|
|왼/오른 무릎|29 / 35|25 / 26|
|왼/오른 발목|31 / 37|27 / 28|

출처: [UI-PRMD 원 논문, Table 2 및 데이터 수집 설명](https://www.mdpi.com/2306-5729/3/1/2), [데이터 제공 페이지](https://www.idahofallshighered.org/vakanski/ui-prmd.html), [MediaPipe Pose Landmarker](https://developers.google.com/edge/mediapipe/solutions/vision/pose_landmarker).
Vicon m03_s03은 원 데이터에서 누락되어 선택 클래스는 총 49개 녹화입니다. Vicon의 100 Hz를 30 Hz로 낮춰 MediaPipe 스트림을 가정합니다. MediaPipe에서는 이미지 정규화 좌표가 아닌 **pose_world_landmarks의 XYZ**를 보내세요. 센서 차이 때문에 여기서 얻는 정확도는 실제 MediaPipe 정확도가 아닙니다.


In [ ]:
import sys, json, random, re, hashlib, shutil, zipfile, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy.signal import resample_poly
import sklearn
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
    classification_report, ConfusionMatrixDisplay, confusion_matrix)
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
print('Python', sys.version, 'Torch', torch.__version__)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


## 1. 데이터 준비
Google Drive 모드가 기본입니다. 이미 /content/Movements에 압축을 풀었다면 MODE='existing'을 사용합니다. 로컬 Jupyter에서도 DATA_ROOT를 자신의 Movements 경로로 지정할 수 있습니다. 기존 데이터 폴더를 자동 덮어쓰지 않습니다.


In [ ]:
MODE = 'drive'  # 'drive', 'upload', 'existing'
ZIP_PATH = '/content/drive/MyDrive/Movements.zip'
DATA_ROOT = Path('/content/Movements')
OUT = Path('/content/ui_prmd_tcn_output')
OUT.mkdir(parents=True, exist_ok=True)
if MODE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
elif MODE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    zips = [p for p in uploaded if p.lower().endswith('.zip')]
    if len(zips) != 1: raise ValueError('Movements.zip 하나를 업로드하세요.')
    ZIP_PATH = zips[0]
    del uploaded
elif MODE != 'existing':
    raise ValueError('지원하지 않는 MODE')
if MODE != 'existing' and not DATA_ROOT.exists():
    dest = DATA_ROOT.parent.resolve()
    with zipfile.ZipFile(ZIP_PATH) as z:
        for member in z.infolist():
            target = (dest / member.filename).resolve()
            if not target.is_relative_to(dest): raise ValueError('안전하지 않은 ZIP 경로')
        z.extractall(dest)
POSITION_DIR = DATA_ROOT / 'Vicon' / 'Positions'
assert POSITION_DIR.is_dir(), f'폴더를 확인하세요: {POSITION_DIR}'
print('Data:', POSITION_DIR)


## 2. 전처리 및 평가 설정

1. 파일명에서 클래스·피험자를 파싱하고 117열/유한값을 검사합니다. 0 벡터로 기록된 마커는 결측으로 취급합니다. 선택 관절의 결측률이 5%를 넘거나 연속 결측이 0.3초보다 길면 오류로 중단합니다. 짧은 결측만 시간 보간합니다.
2. 100→30 Hz anti-alias resampling 후 90프레임(3초), stride 30프레임(1초)의 창을 만듭니다. 녹화 경계를 넘지 않습니다. 반복 동작 경계 라벨은 없으므로 **반복 1회 분류가 아닌 3초 창 분류**입니다.
3. 각 창의 골반 중점을 원점으로 이동하고, 창 내 어깨–골반 거리 중앙값으로 크기를 정규화합니다. 창 전체의 좌우 골반 방향과 알려진 수직축으로 고정 신체 좌표계를 구성합니다. 몸통 기울기는 보존합니다. Vicon 수직축은 +Z, MediaPipe는 -Y입니다. 몸이 큰 폭으로 회전하는 입력은 별도 검증이 필요합니다.
4. 특징은 12관절 × XYZ = 36개 위치값뿐입니다. 각도·속도는 사용하지 않습니다. 평균/표준편차는 train 창에서만 계산합니다.

**분할:** train s01–s06, validation s07–s08, test s09–s10. 겹치는 창이 다른 split에 들어가지 않게 피험자를 먼저 분리합니다. Validation으로만 모델을 선택하고 test는 마지막 평가에만 씁니다. 2명의 test 결과는 사람 전체에 대한 안정적인 추정이 아니며, 후속 단계에서는 subject-wise 교차검증과 별도 MediaPipe 수집 평가를 권장합니다.

|파라미터|기본값 / 의미|
|---|---|
|window / stride|90 / 30 (30 Hz에서 3초 / 1초)|
|TCN channels|64 × 4 residual blocks|
|kernel / dilation|3 / 1,2,4,8; block당 Conv1d 2개|
|receptive field|61프레임; 전체 창은 global average pooling으로 결합|
|dropout|0.2|
|optimizer|AdamW, lr=0.001, weight_decay=0.0001|
|batch / epochs|64 / 최대 100|
|early stopping|validation macro-F1 15 epoch 무개선|
|loss|train 클래스 빈도 기반 weighted cross entropy|

좌우 동작 차이를 줄이기 위해 train 창에만 50% 확률로 좌우 관절 교환 및 신체 좌우축 반사를 적용합니다.


In [ ]:
CLASSES = ['M01 Deep Squat', 'M03 Inline Lunge', 'M04 Side Lunge',
           'M06 Standing Active Straight Leg Raise', 'M07 Standing Shoulder Abduction']
CODES = ['m01','m03','m04','m06','m07']
JOINTS = ['left_shoulder','right_shoulder','left_elbow','right_elbow',
          'left_wrist','right_wrist','left_hip','right_hip',
          'left_knee','right_knee','left_ankle','right_ankle']
VICON_GROUPS = [[9],[16],[11],[18],[13,14],[20,21],[23,25],[24,26],[28],[34],[30],[36]]
MP_IDS = [11,12,13,14,15,16,23,24,25,26,27,28]
CFG = dict(source_fps=100, target_fps=30, window=90, stride=30,
           channels=64, dilations=[1,2,4,8], kernel=3, dropout=0.2,
           batch_size=64, epochs=100, patience=15, lr=1e-3, weight_decay=1e-4,
           seed=SEED)
SPLITS = {'train':list(range(1,7)), 'val':[7,8], 'test':[9,10]}
assert not (set(SPLITS['train']) & set(SPLITS['val']) or
            set(SPLITS['train']) & set(SPLITS['test']) or
            set(SPLITS['val']) & set(SPLITS['test']))


In [ ]:
PREPROCESS_SOURCE = "import numpy as np\n\nJOINTS = ['left_shoulder','right_shoulder','left_elbow','right_elbow',\n          'left_wrist','right_wrist','left_hip','right_hip',\n          'left_knee','right_knee','left_ankle','right_ankle']\nMP_IDS = [11,12,13,14,15,16,23,24,25,26,27,28]\n\ndef normalize_window(points, up):\n    p = np.asarray(points, dtype=np.float64)\n    if p.ndim != 3 or p.shape[1:] != (12,3) or not np.isfinite(p).all():\n        raise ValueError('Expected finite [time,12,3] points')\n    center = (p[:,6] + p[:,7]) / 2\n    shoulder = (p[:,0] + p[:,1]) / 2\n    scale = np.median(np.linalg.norm(shoulder-center, axis=-1))\n    if scale < 1e-6: raise ValueError('Degenerate torso')\n    vertical = np.asarray(up, dtype=np.float64)\n    vertical /= np.linalg.norm(vertical)\n    lateral = np.median(p[:,7]-p[:,6], axis=0)\n    lateral -= np.dot(lateral, vertical)*vertical\n    if np.linalg.norm(lateral) < 1e-6: raise ValueError('Degenerate hip axis')\n    lateral /= np.linalg.norm(lateral)\n    depth = np.cross(lateral, vertical)\n    basis = np.stack([lateral,vertical,depth], axis=1)\n    return (((p-center[:,None,:]) @ basis) / scale).reshape(len(p),36).astype(np.float32)\n\ndef mediapipe_json_window(payload, window=90, fps=30, visibility_threshold=0.5):\n    # JSON: {coordinate_system: 'mediapipe_world', frames:\n    # [{timestamp_ms: 0, landmarks: [{x,y,z,visibility}, ... 33 items]}, ...]}\n    if payload.get('coordinate_system') != 'mediapipe_world':\n        raise ValueError('Only MediaPipe world landmarks are supported')\n    frames = payload['frames']\n    times = np.asarray([f['timestamp_ms'] for f in frames], dtype=float)/1000\n    if len(times)<2 or not np.isfinite(times).all() or np.any(np.diff(times)<=0):\n        raise ValueError('Timestamps must be finite and strictly increasing')\n    if np.max(np.diff(times)) > 0.2: raise ValueError('Tracking gap > 0.2 seconds')\n    if any(len(f['landmarks']) != 33 for f in frames): raise ValueError('Need 33 landmarks')\n    p = np.asarray([[[f['landmarks'][j][a] for a in ('x','y','z')]\n                     for j in MP_IDS] for f in frames], dtype=float)\n    quality = np.asarray([[f['landmarks'][j].get('visibility',0) for j in MP_IDS]\n                          for f in frames], dtype=float)\n    if not np.isfinite(p).all() or not np.isfinite(quality).all() or np.any(quality<visibility_threshold):\n        raise ValueError('Low visibility or invalid coordinates: collect a new window')\n    target = times[-1] - np.arange(window-1,-1,-1)/fps\n    if target[0] < times[0]-1e-6: raise ValueError('Need at least 3 seconds of history')\n    flat = p.reshape(len(p),-1)\n    sampled = np.stack([np.interp(target,times,flat[:,k]) for k in range(36)],axis=-1)\n    return normalize_window(sampled.reshape(window,12,3), [0,-1,0])\n"
exec(PREPROCESS_SOURCE)
(OUT/'preprocess.py').write_text(PREPROCESS_SOURCE, encoding='utf-8')


In [ ]:
def read_vicon(path):
    first = path.open().readline()
    raw = np.loadtxt(path, delimiter=',' if ',' in first else None, ndmin=2)
    if raw.shape[1] != 117: raise ValueError(f'{path.name}: expected 117 columns, got {raw.shape}')
    p = raw.reshape(-1,39,3)
    chosen = sorted(set(sum(VICON_GROUPS, [])))
    missing = ~np.isfinite(p).all(-1) | np.all(p==0,axis=-1)
    for j in chosen:
        mask = missing[:,j]
        runs = re.findall('1+', ''.join(mask.astype(int).astype(str)))
        if mask.mean()>0.05 or max(map(len,runs),default=0)>30:
            raise ValueError(f'{path.name}: excessive missing marker {j+1}')
        for axis in range(3):
            p[:,j,axis] = np.interp(np.arange(len(p)), np.flatnonzero(~mask), p[~mask,j,axis])
    common = np.stack([p[:,g].mean(1) for g in VICON_GROUPS],axis=1)
    return resample_poly(common,3,10,axis=0).astype(np.float32), len(raw), float(missing[:,chosen].mean())

records, manifest = [], []
for path in sorted(POSITION_DIR.glob('*.txt')):
    match = re.fullmatch(r'(m\d{2})_s(\d{2})_positions.txt', path.name)
    if not match or match[1] not in CODES: continue
    motion, subject = match[1], int(match[2])
    split = next((k for k,v in SPLITS.items() if subject in v), None)
    if split is None: raise ValueError(f'Unknown subject: {subject}')
    p, n_raw, missing_rate = read_vicon(path)
    if len(p)<CFG['window']: raise ValueError(f'Short recording: {path}')
    records.append(dict(points=p, label=CODES.index(motion), subject=subject,
                        split=split, file=path.name))
    manifest.append(dict(file=path.name, subject=subject, motion=motion, split=split,
                         raw_frames=n_raw, frames_30hz=len(p), missing_rate=missing_rate,
                         sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
manifest = pd.DataFrame(manifest)
expected = {(m,s) for m in CODES for s in range(1,11)} - {('m03',3)}
actual = {(r['motion'],r['subject']) for r in manifest.to_dict('records')}
assert actual == expected, f'Dataset mismatch. Missing: {expected-actual}; extra: {actual-expected}'
display(manifest.groupby(['split','motion']).size().unstack(fill_value=0))
manifest.to_csv(OUT/'data_manifest.csv',index=False)

X, Y, META = {}, {}, {}
for split in SPLITS:
    xs, ys, ms = [], [], []
    for r in records:
        if r['split'] != split: continue
        for start in range(0,len(r['points'])-CFG['window']+1,CFG['stride']):
            xs.append(normalize_window(r['points'][start:start+CFG['window']], [0,0,1]))
            ys.append(r['label'])
            ms.append(dict(file=r['file'],subject=r['subject'],start_frame=start))
    X[split], Y[split], META[split] = np.stack(xs), np.array(ys), pd.DataFrame(ms)
    assert set(Y[split]) == set(range(5))
    print(split, X[split].shape, np.bincount(Y[split],minlength=5))
mean = X['train'].mean(axis=(0,1),keepdims=True)
std = np.maximum(X['train'].std(axis=(0,1),keepdims=True), 1e-3)
np.savez(OUT/'normalization.npz', mean=mean, std=std)
fig, ax = plt.subplots(figsize=(8,3))
pd.DataFrame({s:np.bincount(Y[s],minlength=5) for s in SPLITS},index=CODES).plot.bar(ax=ax)
ax.set(title='Window counts by subject split',ylabel='Windows'); plt.tight_layout(); plt.show()


### 학습 전 좌표 시각화
각 동작의 중앙 시점 골격을 확인합니다. X=신체 좌우, Y=수직, Z=깊이입니다. 마커 매핑이나 데이터가 바뀌면 이 그림을 먼저 확인하세요.


In [ ]:
EDGES = [(0,1),(0,2),(2,4),(1,3),(3,5),(0,6),(1,7),(6,7),(6,8),(8,10),(7,9),(9,11)]
fig = plt.figure(figsize=(16,3))
for label in range(5):
    q = X['train'][np.flatnonzero(Y['train']==label)[0],45].reshape(12,3)
    ax = fig.add_subplot(1,5,label+1,projection='3d')
    for a,b in EDGES: ax.plot(q[[a,b],0],q[[a,b],2],q[[a,b],1])
    ax.set(title=CODES[label],xlabel='Lateral',ylabel='Depth',zlabel='Up')
    ax.set_box_aspect((1,1,1))
plt.tight_layout(); plt.savefig(OUT/'skeleton_preview.png',dpi=160); plt.show()


## 3. TCN 모델과 학습
입력 [batch,90,36] → 내부 표준화 → [batch,36,90] → causal dilated residual Conv1d → 시간 평균 pooling → 5 logits. causal convolution은 각 시점에서 미래를 보지 않지만 최종 출력은 수집된 3초 창 전체를 사용합니다. LayerNorm은 시점별 채널에 적용합니다. 저장 모델 내부에 train 평균·표준편차가 포함되므로 서버에서 표준화를 중복 적용하지 않습니다.


In [ ]:
MODEL_SOURCE = "import torch\nfrom torch import nn\nimport torch.nn.functional as F\n\nclass TemporalBlock(nn.Module):\n    def __init__(self, cin, cout, dilation, kernel, dropout):\n        super().__init__()\n        self.pad = (kernel-1)*dilation\n        self.c1 = nn.Conv1d(cin,cout,kernel,dilation=dilation)\n        self.c2 = nn.Conv1d(cout,cout,kernel,dilation=dilation)\n        self.n1 = nn.LayerNorm(cout)\n        self.n2 = nn.LayerNorm(cout)\n        self.drop = nn.Dropout(dropout)\n        self.skip = nn.Conv1d(cin,cout,1) if cin!=cout else nn.Identity()\n    def forward(self,x):\n        z = self.c1(F.pad(x,(self.pad,0)))\n        z = self.drop(F.relu(self.n1(z.transpose(1,2)).transpose(1,2)))\n        z = self.c2(F.pad(z,(self.pad,0)))\n        z = self.drop(F.relu(self.n2(z.transpose(1,2)).transpose(1,2)))\n        return F.relu(z+self.skip(x))\n\nclass PoseTCN(nn.Module):\n    def __init__(self,cfg,mean,std):\n        super().__init__()\n        self.register_buffer('mean',torch.as_tensor(mean,dtype=torch.float32))\n        self.register_buffer('std',torch.as_tensor(std,dtype=torch.float32))\n        blocks=[]\n        for i,d in enumerate(cfg['dilations']):\n            blocks.append(TemporalBlock(36 if i==0 else cfg['channels'],cfg['channels'],\n                                        d,cfg['kernel'],cfg['dropout']))\n        self.blocks=nn.Sequential(*blocks)\n        self.head=nn.Linear(cfg['channels'],5)\n    def forward(self,x):\n        x=((x-self.mean)/self.std).transpose(1,2)\n        return self.head(self.blocks(x).mean(dim=-1))\n"
exec(MODEL_SOURCE)
(OUT/'model.py').write_text(MODEL_SOURCE,encoding='utf-8')
model = PoseTCN(CFG,mean,std).to(DEVICE)
print('Parameters:',sum(p.numel() for p in model.parameters()))
loaders = {s:DataLoader(TensorDataset(torch.from_numpy(X[s]),torch.from_numpy(Y[s]).long()),
                       batch_size=CFG['batch_size'],shuffle=(s=='train'),num_workers=0)
           for s in SPLITS}
counts=np.bincount(Y['train'],minlength=5)
weights=len(Y['train'])/(5*counts)
criterion=nn.CrossEntropyLoss(weight=torch.tensor(weights,dtype=torch.float32,device=DEVICE))
optimizer=torch.optim.AdamW(model.parameters(),lr=CFG['lr'],weight_decay=CFG['weight_decay'])

def augment(x):
    p=x.reshape(-1,CFG['window'],12,3).clone()
    mask=torch.rand(len(p),device=p.device)<0.5
    mirrored=p[mask][:,:,[1,0,3,2,5,4,7,6,9,8,11,10],:].clone()
    mirrored[:,:,:,0]*=-1
    p[mask]=mirrored
    return p.reshape(-1,CFG['window'],36)

def run_epoch(loader, training=False):
    model.train(training)
    losses=[]; truths=[]; probs=[]
    for x,y in loader:
        x,y=x.to(DEVICE),y.to(DEVICE)
        if training: x=augment(x)
        with torch.set_grad_enabled(training):
            logits=model(x); loss=criterion(logits,y)
            if training:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(),1.0)
                optimizer.step()
        losses.append((loss.item(),len(y)))
        truths.extend(y.cpu().numpy()); probs.append(logits.detach().softmax(-1).cpu().numpy())
    truth=np.asarray(truths); prob=np.concatenate(probs); pred=prob.argmax(1)
    return dict(loss=sum(l*n for l,n in losses)/sum(n for _,n in losses),
                accuracy=accuracy_score(truth,pred),macro_f1=f1_score(truth,pred,average='macro'),
                truth=truth,prob=prob)

history=[]; best=-1.; stale=0
for epoch in range(1,CFG['epochs']+1):
    tr=run_epoch(loaders['train'],True); va=run_epoch(loaders['val'])
    history.append(dict(epoch=epoch,**{'train_'+k:tr[k] for k in ['loss','accuracy','macro_f1']},
                        **{'val_'+k:va[k] for k in ['loss','accuracy','macro_f1']}))
    if va['macro_f1']>best+1e-5:
        best=va['macro_f1']; stale=0; best_epoch=epoch
        torch.save(model.state_dict(),OUT/'best_state_dict.pt')
    else: stale+=1
    print(f"epoch {epoch:03d} | train loss {tr['loss']:.4f} | val F1 {va['macro_f1']:.4f} | best {best:.4f}")
    if stale>=CFG['patience']: break
model.load_state_dict(torch.load(OUT/'best_state_dict.pt',map_location=DEVICE,weights_only=True))
history=pd.DataFrame(history)
history.to_csv(OUT/'history.csv',index=False)
fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,metric in zip(axes,['loss','accuracy','macro_f1']):
    ax.plot(history.epoch,history['train_'+metric],label='train (augmented)')
    ax.plot(history.epoch,history['val_'+metric],label='validation')
    ax.axvline(best_epoch,color='gray',linestyle=':'); ax.set_title(metric); ax.set_xlabel('Epoch'); ax.legend()
plt.tight_layout(); plt.savefig(OUT/'learning_curves.png',dpi=160); plt.show()


## 4. 최종 평가와 성능 수준 해석
Accuracy와 macro-F1(클래스별 F1 평균), balanced accuracy, 클래스별 precision/recall, confusion matrix 및 사람별 성능을 확인합니다. 균등 무작위 예측 기대 accuracy는 20%입니다. 이 값을 넘었다고 서비스 준비가 완료된 것은 아닙니다.

겹치는 창은 독립 표본이 아니므로 창 개수로 신뢰구간을 과장하지 않습니다. 녹화 단위 평균 확률 결과도 함께 제시하지만 이는 여러 반복을 포함한 전체 녹화를 본 결과라 실시간 성능과 구분해야 합니다. validation 대비 test 하락, 특정 클래스 혼동, 사람 간 편차로 일반화 수준을 판단하세요. 실제 MediaPipe Lite 영상의 별도 test가 배포 판단의 기준입니다.


In [ ]:
result=run_epoch(loaders['test'])
truth=result['truth']; prob=result['prob']; pred=prob.argmax(1)
metrics=dict(window_accuracy=float(accuracy_score(truth,pred)),
             window_macro_f1=float(f1_score(truth,pred,average='macro')),
             window_balanced_accuracy=float(balanced_accuracy_score(truth,pred)),
             best_validation_macro_f1=float(best),best_epoch=best_epoch)
print(json.dumps(metrics,indent=2))
report=classification_report(truth,pred,labels=list(range(5)),target_names=CLASSES,
                             output_dict=True,zero_division=0)
display(pd.DataFrame(report).T)
pd.DataFrame(report).T.to_csv(OUT/'classification_report.csv')
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,norm in zip(axes,[None,'true']):
    ConfusionMatrixDisplay.from_predictions(truth,pred,labels=range(5),display_labels=CODES,
        normalize=norm,values_format='.2f' if norm else 'd',ax=ax,colorbar=False)
    ax.set_title('Test confusion: '+('row normalized' if norm else 'counts'))
plt.tight_layout(); plt.savefig(OUT/'confusion_matrix.png',dpi=160); plt.show()
details=META['test'].copy(); details['true']=truth; details['pred']=pred
details['confidence']=prob.max(1)
for i,c in enumerate(CODES): details[c+'_prob']=prob[:,i]
details.to_csv(OUT/'test_predictions.csv',index=False)
subject_scores=[]
for subject,group in details.groupby('subject'):
    subject_scores.append(dict(subject=subject,accuracy=accuracy_score(group.true,group.pred),
        macro_f1=f1_score(group.true,group.pred,labels=range(5),average='macro',zero_division=0)))
subject_scores=pd.DataFrame(subject_scores)
display(subject_scores); subject_scores.to_csv(OUT/'subject_scores.csv',index=False)
record_scores=details.groupby('file')[[c+'_prob' for c in CODES]].mean()
record_truth=details.groupby('file').true.first().loc[record_scores.index]
metrics['recording_accuracy']=float(accuracy_score(record_truth,record_scores.values.argmax(1)))
metrics['test_subjects']=SPLITS['test']
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].bar(['Uniform chance','Test accuracy','Test macro-F1'],[0.2,metrics['window_accuracy'],metrics['window_macro_f1']])
axes[0].set_ylim(0,1); axes[0].set_title('Window-level performance')
subject_scores.set_index('subject').plot.bar(ax=axes[1],ylim=(0,1),title='Held-out subjects')
plt.tight_layout(); plt.savefig(OUT/'performance_summary.png',dpi=160); plt.show()
(OUT/'metrics.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')
print('Recording accuracy (not real-time):',metrics['recording_accuracy'])


## 5. 모델 저장·재로드 검증·다운로드
출력 ZIP에는 TorchScript 모델, state_dict, 모델 정의, 공통 전처리 함수, 관절 매핑/파라미터/클래스 순서, train 정규화 값, 데이터 해시/분할, 평가 CSV 및 그림이 포함됩니다. TorchScript 입력은 **신체 좌표 정규화를 마친 [B,90,36] float32**, 출력은 softmax 전의 5 logits입니다. 표준화는 모델 내부에서 수행됩니다.

Colab 임시 저장소는 종료하면 사라집니다. 다운로드 셀 또는 Drive 복사 옵션으로 보관하세요.


In [ ]:
model=model.cpu().eval()
example=torch.from_numpy(X['test'][:2])
traced=torch.jit.trace(model,example)
traced.save(str(OUT/'pose_tcn.torchscript.pt'))
loaded=torch.jit.load(str(OUT/'pose_tcn.torchscript.pt')).eval()
with torch.no_grad():
    for n in [1,2]:
        torch.testing.assert_close(loaded(example[:n]),model(example[:n]),atol=1e-5,rtol=1e-5)
metadata=dict(config=CFG,classes=CLASSES,codes=CODES,joints=JOINTS,
              vicon_marker_groups_zero_based=VICON_GROUPS,mediapipe_indices=MP_IDS,
              splits=SPLITS,input_shape=['batch',90,36],input_dtype='float32',
              input='body-normalized positions; standardization is inside model',
              vicon_up=[0,0,1],mediapipe_up=[0,-1,0],
              source='Vicon Positions only; marker proxies are not anatomical joint centers',
              versions=dict(python=platform.python_version(),torch=torch.__version__,
                  numpy=np.__version__,scipy=scipy.__version__,sklearn=sklearn.__version__))
(OUT/'metadata.json').write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding='utf-8')
print('Reload equivalence passed. Artifacts:',OUT)
archive=shutil.make_archive(str(OUT),'zip',OUT)
print(archive)
COPY_TO_DRIVE = True
if COPY_TO_DRIVE and Path('/content/drive/MyDrive').is_dir():
    shutil.copy2(archive,'/content/drive/MyDrive/ui_prmd_tcn_output.zip')
    print('Copied to Google Drive')


In [ ]:
# Colab에서 PC로 다운로드 (선택)
DOWNLOAD = True
if DOWNLOAD:
    try:
        from google.colab import files
        files.download(archive)
    except ImportError:
        print('Local Jupyter: open',archive)


## 6. 서버에서 MediaPipe JSON으로 추론하는 예시
아래 코드는 학습과 분리된 서버용 예시입니다. 압축을 푼 폴더를 ARTIFACT_DIR로 설정하세요. **한 프레임으로는 TCN 분류를 하지 않습니다.** 약 3초의 timestamp가 있는 world landmarks를 버퍼에 모아 사용합니다. 최신 90프레임을 30 Hz로 보간합니다. 추적 끊김/낮은 visibility는 분류를 보류합니다. 카메라 미러링으로 좌우 라벨을 바꾸지 마세요.

예상 JSON: coordinate_system='mediapipe_world', frames=[{timestamp_ms: ..., landmarks: [{x: ..., y: ..., z: ..., visibility: ...}, ... 33개]}, ...]. timestamp는 밀리초 단조 증가 값입니다. 클라이언트에서 보내기 전에 약 30 Hz로 맞추는 것이 좋습니다. 5개 외의 동작이나 정지 자세도 강제로 하나로 분류되는 closed-set 모델입니다. 최대 softmax 확률은 보정된 신뢰도나 동작 품질 점수가 아닙니다. 추후 background/unknown 데이터 및 보류 임계값을 별도 validation으로 정해야 합니다.


In [ ]:
# 실제 JSON이 준비되었을 때만 실행됩니다.
JSON_PATH = None  # 예: '/content/mediapipe_window.json'
if JSON_PATH is not None:
    payload=json.loads(Path(JSON_PATH).read_text(encoding='utf-8'))
    features=mediapipe_json_window(payload,CFG['window'],CFG['target_fps'])
    serving_model=torch.jit.load(str(OUT/'pose_tcn.torchscript.pt')).eval()
    with torch.no_grad():
        scores=serving_model(torch.from_numpy(features[None])).softmax(-1)[0].numpy()
    print({'class':CLASSES[int(scores.argmax())],
           'softmax_score':float(scores.max()),'probabilities':scores.tolist()})


### 다음 실험
현재 결과는 Vicon에서 학습하고 새로운 Vicon 피험자에게 평가한 baseline입니다. MediaPipe Lite로 동일 5동작을 여러 사람·촬영각도·거리에서 수집하고 사람 단위로 분할하여 transfer 성능을 먼저 측정하세요. 그 후 MediaPipe 데이터 fine-tuning, background 클래스, 좌우/가림 조건, 지연시간을 평가할 수 있습니다. train/validation 성능 차이가 크면 모델 크기·dropout·창 길이를 validation으로 조정하며 test를 반복 튜닝에 사용하지 마세요.
